# Notebook 3B — Matching, Difference-in-Differences & Event Study
### Study: *The Second-Order Compute Footprint of AI-Coding-Tool Adoption in Open Source*

Runs on the panels from 3A. **No data collection** — pure analysis, fast and fully reproducible. Answers **RQ1** (activity effect) and **RQ3** (heterogeneity), and produces the CI-minutes DiD effect that Notebook 4 converts to carbon (**RQ2**).

**Outcome set (Option B; churn dropped — deviation logged):** commit frequency, PR volume, CI-runs (activity); CI-minutes (primary compute outcome, CI-eligible subset).

**Pipeline**
1. **Matching** — propensity-score / nearest-neighbor matching of controls to treated on pre-adoption covariates; **covariate-balance (SMD) table** (target |SMD| < 0.1).
2. **DiD** — two-way fixed-effects (repo + calendar-month), `treated×post` interaction, cluster-robust SE by repo; outcomes on **log1p** scale (right-skewed counts) so coefficients read as approximate % changes.
3. **Event study** — leads/lags around adoption + joint pre-trend test (the parallel-trends check that licenses the causal reading).
4. **Multiplicity** — Benjamini–Hochberg across outcomes.
5. **Placebo** — random pseudo-adoption dates must yield null.

**Standards:** ACM SIGSOFT *Data Science* (predictive/inferential modeling, validity), *Longitudinal*. Effect sizes + 95% CIs lead; p-values secondary.


---
## Cell 0 — Re-attach environment & load panels


In [ ]:

%pip install -q pandas pyarrow numpy statsmodels scipy matplotlib

import json, pathlib, numpy as np, pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/ai_coding_carbon')
DIRS = {k: PROJECT_ROOT/k for k in ['panels','cohort','results','figures','logs']}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
CONFIG = json.loads((PROJECT_ROOT/'config.json').read_text())
np.random.seed(CONFIG['random_seed'])

panel = pd.read_parquet(DIRS['panels']/'panel_full.parquet')
cohort = pd.read_parquet(DIRS['cohort']/'cohort.parquet')
print('panel rows:', len(panel), '| repos:', panel['full_name'].nunique(),
      '| treated:', panel[panel['is_treated']]['full_name'].nunique())

OUTCOMES = ['commits','prs','ci_runs','ci_minutes']
print('Outcomes:', OUTCOMES)


---
## Cell 1 — Pre-adoption covariates for matching

We compute each repo's **pre-adoption** characteristics (baseline activity in the pre-window, plus static features: stars, language, band, age). Matching on these makes treated and control comparable on everything that predicts both adoption and the outcomes, so the DiD isolates the adoption effect rather than selection.


In [ ]:

panel['adoption_date'] = pd.to_datetime(panel['adoption_date'], utc=True)
pre = panel[panel['event_month'].between(-6,-1)]

base = (pre.groupby('full_name')
          .agg(pre_commits=('commits','mean'),
               pre_prs=('prs','mean'),
               pre_ci_minutes=('ci_minutes','mean'),
               pre_ci_runs=('ci_runs','mean'))
          .reset_index())

meta = cohort[['full_name','language','band','is_treated','stars',
               'adoption_date','ci_eligible']].drop_duplicates('full_name').copy()
meta['adoption_date'] = pd.to_datetime(meta['adoption_date'], utc=True)
# repo age (months) at adoption — needs created_at from frame
frame = pd.read_parquet(PROJECT_ROOT/'frame'/'frame_final.parquet')[['full_name','created_at']]
meta = meta.merge(frame, on='full_name', how='left')
meta['created_at'] = pd.to_datetime(meta['created_at'], utc=True)
meta['age_months'] = ((meta['adoption_date']-meta['created_at']).dt.days/30.44).clip(lower=0)

cov = meta.merge(base, on='full_name', how='left').fillna(
    {'pre_commits':0,'pre_prs':0,'pre_ci_minutes':0,'pre_ci_runs':0})
cov['log_stars']=np.log1p(cov['stars'])
cov['log_pre_commits']=np.log1p(cov['pre_commits'])
cov['log_pre_prs']=np.log1p(cov['pre_prs'])
cov.to_parquet(DIRS['cohort']/'covariates.parquet', index=False)
print('Covariate table:', cov.shape)
print(cov[['is_treated','pre_commits','pre_prs','log_stars','age_months']]
      .groupby('is_treated').mean())


---
## Cell 2 — Propensity-score nearest-neighbor matching + SMD balance

We estimate a propensity model (P(treated) on the covariates), then match each treated repo to its nearest control(s) by propensity (with a caliper). We report **standardized mean differences (SMD)** before vs. after matching; |SMD| < 0.1 indicates good balance. Unmatched treated (no control within caliper) are reported.


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

cov = pd.read_parquet(DIRS['cohort']/'covariates.parquet')
feat = ['log_stars','log_pre_commits','log_pre_prs','age_months']
lang_d = pd.get_dummies(cov['language'], prefix='lang')
X = pd.concat([cov[feat], lang_d], axis=1).fillna(0).astype(float)
y = cov['is_treated'].astype(int).values

Xs = StandardScaler().fit_transform(X)
ps = LogisticRegression(max_iter=1000).fit(Xs, y).predict_proba(Xs)[:,1]
cov['pscore']=ps

treated = cov[cov['is_treated']].copy()
control = cov[~cov['is_treated']].copy()
caliper = 0.2*np.std(np.log(cov['pscore']/(1-cov['pscore'])).replace([np.inf,-np.inf],np.nan).dropna())

# greedy NN match (k=2 controls per treated, with replacement) within caliper
from scipy.spatial import cKDTree
ctree = cKDTree(control[['pscore']].values)
matches=[]
for _,t in treated.iterrows():
    d,idx = ctree.query([t['pscore']], k=min(2,len(control)))
    d=np.atleast_1d(d); idx=np.atleast_1d(idx)
    for dd,ii in zip(d,idx):
        if dd<=max(caliper,0.05):
            matches.append((t['full_name'], control.iloc[ii]['full_name']))
matched_treated=set(m[0] for m in matches)
matched_control=set(m[1] for m in matches)
keep = matched_treated | matched_control
cov['matched']=cov['full_name'].isin(keep)

def smd(a,b):
    sa,sb=a.std(),b.std(); sp=np.sqrt((sa**2+sb**2)/2)
    return (a.mean()-b.mean())/sp if sp>0 else 0.0

rows=[]
for f in feat:
    pre_s = smd(treated[f], control[f])
    mt=cov[(cov['is_treated'])&(cov['matched'])][f]
    mc=cov[(~cov['is_treated'])&(cov['matched'])][f]
    rows.append({'covariate':f,'SMD_before':round(pre_s,3),'SMD_after':round(smd(mt,mc),3)})
bal=pd.DataFrame(rows)
bal.to_csv(DIRS['results']/'balance_table.csv', index=False)
cov.to_parquet(DIRS['cohort']/'covariates_matched.parquet', index=False)
print(f"Matched treated: {len(matched_treated)} | matched controls: {len(matched_control)} "
      f"| unmatched treated: {len(treated)-len(matched_treated)}")
print("\nCovariate balance (|SMD|<0.1 = good):"); print(bal.to_string(index=False))


---
## Cell 3 — DiD on the matched sample (all outcomes)

Two-way fixed-effects DiD: `log1p(outcome) ~ treated:post + C(full_name) + C(ym)`, cluster-robust SE by repo. The `treated:post` coefficient is the DiD estimate; on the log1p scale, `exp(β)-1` ≈ the proportional change attributable to adoption. CI-minutes/CI-runs restricted to CI-eligible repos.


In [ ]:

cov = pd.read_parquet(DIRS['cohort']/'covariates_matched.parquet')
matched_names = set(cov[cov['matched']]['full_name'])
p = panel[panel['full_name'].isin(matched_names)].copy()
p = p[p['event_month'].between(-6,5)].copy()        # windowed
p['post'] = (p['event_month']>=0).astype(int)
p['treated'] = p['is_treated'].astype(int)
for o in OUTCOMES: p[f'log_{o}']=np.log1p(p[o])

def run_did(df, outcome, eligible_only=False):
    d = df.copy()
    if eligible_only: d = d[d['ci_eligible']]
    d = d[d['full_name'].map(d.groupby('full_name')['post'].nunique())>=1]  # keep repos w/ obs
    model = smf.ols(f'log_{outcome} ~ treated:post + C(full_name) + C(ym)', data=d)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d['full_name']})
    key=[k for k in res.params.index if 'treated:post' in k]
    if not key: return None
    k=key[0]; beta=res.params[k]; se=res.bse[k]
    ci=res.conf_int().loc[k].values
    return {'outcome':outcome,'beta':beta,'se':se,'ci_low':ci[0],'ci_high':ci[1],
            'pct_change':(np.exp(beta)-1)*100,
            'pct_ci_low':(np.exp(ci[0])-1)*100,'pct_ci_high':(np.exp(ci[1])-1)*100,
            'pvalue':res.pvalues[k],'n_obs':int(d.shape[0]),
            'n_repos':int(d['full_name'].nunique())}

results=[]
for o in ['commits','prs']:
    r=run_did(p,o); 
    if r: results.append(r)
for o in ['ci_runs','ci_minutes']:
    r=run_did(p,o,eligible_only=True)
    if r: results.append(r)

res_df=pd.DataFrame(results)
# Benjamini-Hochberg across outcomes
from statsmodels.stats.multitest import multipletests
if len(res_df):
    res_df['p_bh']=multipletests(res_df['pvalue'],method='fdr_bh')[1]
res_df.to_csv(DIRS['results']/'did_main.csv', index=False)
print("MAIN DiD RESULTS (log1p scale; pct_change = exp(beta)-1)")
print(res_df[['outcome','pct_change','pct_ci_low','pct_ci_high','pvalue','p_bh',
              'n_repos','n_obs']].round(2).to_string(index=False))


---
## Cell 4 — Event study & parallel-trends check (primary outcome: CI-minutes)

We estimate leads/lags relative to adoption. **Flat, non-significant pre-adoption leads** support the parallel-trends assumption that licenses the causal reading; post-adoption coefficients trace the effect's emergence. We plot the path with 95% CIs and run a joint test that pre-period leads = 0.


In [ ]:

def event_study(df, outcome, eligible_only=False):
    d=df.copy()
    if eligible_only: d=d[d['ci_eligible']]
    d=d[d['event_month'].between(-6,5)].copy()
    d['et']=d['event_month'].astype(int)
    d=d[d['et']!=-1]                                  # omit -1 as reference
    d['etf']=pd.Categorical(d['et'])
    res=smf.ols(f'log_{outcome} ~ treated:C(etf) + C(full_name) + C(ym)', data=d)\
          .fit(cov_type='cluster', cov_kwds={'groups':d['full_name']})
    coefs=[]
    for k in res.params.index:
        if 'treated:C(etf)' in k:
            et=int(k.split('[T.')[-1].rstrip(']'))
            ci=res.conf_int().loc[k].values
            coefs.append({'event_month':et,'beta':res.params[k],
                          'ci_low':ci[0],'ci_high':ci[1],'p':res.pvalues[k]})
    ev=pd.DataFrame(coefs).sort_values('event_month')
    # joint pre-trend test (leads < -1)
    leads=[k for k in res.params.index if 'treated:C(etf)' in k and
           int(k.split('[T.')[-1].rstrip(']'))< -1]
    jt=res.f_test(' = 0, '.join(leads)+' = 0') if len(leads)>1 else None
    return ev, jt

ev, jt = event_study(p,'ci_minutes',eligible_only=True)
ev.to_csv(DIRS['results']/'event_study_ci_minutes.csv', index=False)

plt.figure(figsize=(8,4.5))
plt.axhline(0,color='gray',lw=.8); plt.axvline(-0.5,color='red',ls='--',lw=.8,label='adoption')
plt.errorbar(ev['event_month'],ev['beta'],
             yerr=[ev['beta']-ev['ci_low'],ev['ci_high']-ev['beta']],
             fmt='o-',capsize=3)
plt.xlabel('months relative to adoption'); plt.ylabel('DiD coef (log CI-minutes)')
plt.title('Event study: CI-minutes around AI-tool adoption'); plt.legend()
plt.tight_layout(); plt.savefig(DIRS['figures']/'event_study_ci_minutes.png',dpi=150)
plt.show()
print("Pre-trend joint test (leads=0):", 
      f"F={jt.fvalue:.2f}, p={jt.pvalue:.3f}" if jt is not None else "n/a (too few leads)")
print("  -> p>0.05 supports parallel trends (good).")


---
## Cell 5 — Heterogeneity (RQ3) & placebo test

**RQ3:** re-estimate the CI-minutes DiD within strata (band, language) → forest plot. Java/thin strata flagged as underpowered (wide CIs), not over-claimed.

**Placebo:** assign random fake adoption dates to *control* repos and run the DiD; a credible design yields a null placebo effect (no spurious treatment effect where none exists).


In [ ]:

# ---- heterogeneity by band ----
het=[]
for band in ['high','mid','low']:
    sub=p[p['band']==band]
    r=run_did(sub,'ci_minutes',eligible_only=True)
    if r: r['stratum']=f'band={band}'; het.append(r)
for lang in p['language'].unique():
    sub=p[p['language']==lang]
    r=run_did(sub,'ci_minutes',eligible_only=True)
    if r: r['stratum']=f'lang={lang}'; het.append(r)
het_df=pd.DataFrame(het)
het_df.to_csv(DIRS['results']/'heterogeneity_ci_minutes.csv', index=False)
print("Heterogeneity (CI-minutes, % change):")
print(het_df[['stratum','pct_change','pct_ci_low','pct_ci_high','n_repos']].round(1).to_string(index=False))

# ---- placebo: random fake adoption among controls only ----
ctrl_panel = panel[(~panel['is_treated'])].copy()
rng=np.random.default_rng(CONFIG['random_seed'])
fake = ctrl_panel.copy()
# randomly split controls into pseudo-treated/pseudo-control
names=fake['full_name'].unique()
pseudo_t=set(rng.choice(names,size=len(names)//2,replace=False))
fake['treated']=fake['full_name'].isin(pseudo_t).astype(int)
fake['post']=(fake['event_month']>=0).astype(int)
fake=fake[fake['event_month'].between(-6,5)]
fake['log_ci_minutes']=np.log1p(fake['ci_minutes'])
fake=fake[fake['ci_eligible']]
try:
    pl=smf.ols('log_ci_minutes ~ treated:post + C(full_name) + C(ym)',data=fake)\
        .fit(cov_type='cluster',cov_kwds={'groups':fake['full_name']})
    k=[x for x in pl.params.index if 'treated:post' in x][0]
    print(f"\nPLACEBO (controls only, fake adoption): beta={pl.params[k]:.3f}, p={pl.pvalues[k]:.3f}")
    print("  -> p>0.05 (null) is the desired result: no spurious effect.")
except Exception as e:
    print("placebo failed:",e)


---
## Cell 6 — Manifest & handoff to Notebook 4


In [ ]:

import datetime
manifest={"notebook":"03B_matching_did_eventstudy",
  "produced":{
    "results/balance_table.csv":(DIRS['results']/'balance_table.csv').exists(),
    "results/did_main.csv":(DIRS['results']/'did_main.csv').exists(),
    "results/event_study_ci_minutes.csv":(DIRS['results']/'event_study_ci_minutes.csv').exists(),
    "results/heterogeneity_ci_minutes.csv":(DIRS['results']/'heterogeneity_ci_minutes.csv').exists(),
    "figures/event_study_ci_minutes.png":(DIRS['figures']/'event_study_ci_minutes.png').exists(),
  },"timestamp":datetime.datetime.utcnow().isoformat()+"Z"}
(PROJECT_ROOT/'manifest_nb03B.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
print("\nNotebook 4 (next): convert the CI-minutes DiD effect to energy/CO2e via")
print("Green Algorithms / SCI (ISO/IEC 21031:2024) / CCF factors, reported as a")
print("low/central/high sensitivity band (RQ2 headline carbon figure).")
